# Differential gene expression

In [1]:
# Path and system utilities
import os                    # Operating system interface
import sys                   # System-specific parameters and functions
import glob                  # File pattern matching
from pathlib import Path     # Object-oriented filesystem paths
from pyhere import here      # Reproducible project paths
import gc
import time

# Single-cell data handling
import anndata as ad            # Core data structure for single-cell data
import scanpy as sc

# milo
import pertpy as pt
milo = pt.tl.Milo()
import mudata as md
from mudata import MuData

# Parallel processing
from joblib import Parallel, delayed, parallel_backend


# dataframes
import pandas as pd
import numpy as np
from collections import defaultdict
# plotting
import matplotlib.pyplot as plt 

# Diff genes
from flash_mm import lmmfit, lmmtest, contrast_matrix, sslmm, lmm
from statsmodels.stats.multitest import multipletests

# Custom modules and functions
sys.path.append(str(here('scripts/misc')))  # Add custom script path to system
import diff_genes as dg
import misc as mi
import dg_milo as dm

/work/islet_cartography_scrna/scrna_cartography_milo/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Paths
base_dir = str(here('data/differential_abundance/'))
plot_dir = os.path.join(base_dir, 'plot') 
files_dir = os.path.join(base_dir, 'files') 
de_dir = os.path.join(base_dir, 'de_analysis_groups') 
de_overall_dir = os.path.join(base_dir, 'de_analysis_overall') 
objects_dir = os.path.join(base_dir, 'objects') 
anndata_dir = str(here('data/anndata/'))

In [3]:
mdata =  md.read(os.path.join(objects_dir, "mdata_666_annotation.h5mu"), backed='r')

## Functions

In [4]:
def get_comparisons(anno_direction_values):
    """Build list of (c1, c2, label) comparisons from anno_direction values."""
    # Only keep up/down clusters
    clusters = [c for c in anno_direction_values.unique()
                if isinstance(c, str) and c.endswith(("_up", "_down"))]

    # Build comparisons: prefer up vs down, fallback to vs other
    groups = defaultdict(dict)
    for c in clusters:
        base, direction = c.rsplit("_", 1)
        groups[base][direction] = c
    
    comparisons = []
    for base, g in groups.items():
        if "up" in g and "down" in g:
            comparisons.append((g["up"], g["down"], f"{base}_up_vs_down"))
        else:
            for c in g.values():
                comparisons.append((c, "other", f"{c}_vs_other"))
    return comparisons

def get_subset(adata, df_sub, comp_label, c1, c2):
    """Subset adata to cells in comparison and attach group labels."""
    labels = df_sub["anno_direction"].map(
        lambda x: x if x in [c1, c2] else ("other" if c2 == "other" else None)
    ).rename(comp_label)

    subset = adata[adata.obs_names.isin(df_sub.index)].to_memory()
    subset.obs = subset.obs.join(labels)
    return subset[subset.obs[comp_label].notna()].copy()

## Differential gene expression

#### Map cells to each neighborhood

In [5]:
adata            = mdata["rna"]
nhoods           = adata.obsm["nhoods"]                          
ref_cells        = adata.obs_names[adata.obs["nhood_ixs_refined"] == 1]

rows, cols = nhoods.nonzero()
df = pd.DataFrame({
    "cell"       : adata.obs_names[rows],
    "nhood_id"   : cols,
    "index_cell" : ref_cells[cols],
})

#### Combine differential abundance and cell data

In [6]:
# ----------------------------- SET UP -----------------------------------
print("Setting up variables")
prefix           = "t2d_vs_nd"
out_dir          = de_overall_dir 
da_results_path  = os.path.join(files_dir, f"{prefix}.csv")

nhood_anno_key   = "nhood_annotation"
sample_key       = "ic_id_platform_adjusted_sample"
donor_key        = "ic_id_donor_overall"
disease_key      = "disease_harmonized"

# ----------------------------- LOAD -------------------------------------
da = pd.read_csv(da_results_path, index_col=0)
# ----------------------------- PREPROCESS -------------------------------
print("Combine differential abundance data with per cell neighborhoods")
df_t2d = df.merge(da, how="left")
df_t2d ["direction"] = "no_change"

# Direction of regulations
df_t2d.loc[(df_t2d["SpatialFDR"] <= 0.1) & (df_t2d["logFC"] > 0), "direction"] = "up"
df_t2d.loc[(df_t2d["SpatialFDR"] <= 0.1) & (df_t2d["logFC"] < 0), "direction"] = "down"
df_t2d ["anno_direction"] = df_t2d[nhood_anno_key] + "_" + df_t2d["direction"]

df_t2d = df_t2d .set_index('cell')

Setting up variables
Combine differential abundance data with per cell neighborhoods


#### Differential expression analysis on subset of data

In [ ]:
# Store results per contrast
results = {}
for anno_id in list(df_t2d['nhood_annotation'].drop_duplicates()):
    t0_anno = time.time()
    
    print(f"\nProcessing: {anno_id}")

    # Subset to cell type (annotated neighborhoods)
    df_sub = (df_t2d.loc[df_t2d[nhood_anno_key] == anno_id, ["anno_direction"]]
               .loc[lambda x: ~x.index.duplicated(keep="first")])

    # Define comparisons
    comparisons = get_comparisons(df_sub["anno_direction"])
    
    if not comparisons:
        print(f"  No DA neighborhoods, skipping")
        continue
        
    # Diff for comparison
    for c1, c2, comp_label in comparisons:
        print(f"  DE: {comp_label}")
        
        # Get cells to be used for comparison
        subset = get_subset(adata, df_sub, comp_label, c1, c2)

        # Keep donors that are represented in both groups
        ct = pd.crosstab(subset.obs[donor_key], subset.obs[comp_label])
        donors_keep = ct[(ct > 0).all(axis=1)].index
        
        n_before = ct.shape[0]
        n_after = len(donors_keep)
        
        print(f"  Paired donors: {n_after}/{n_before}")
        
        # Filter subset directly
        subset = subset[subset.obs[donor_key].isin(donors_keep)].copy()


        # Define fixed and random effect
        fixed_effect = comp_label
        random_effect = donor_key

        # Get gene expression (log-transformed counts)(Genes x Cells)
        Y = subset.X.T 
        Y_names = subset.var_names.tolist()

        # Ensure Y is a dense numpy array if it's currently sparse
        if hasattr(Y, "toarray"):
            Y = Y.toarray()
            
        # Filter genes - remove genes that are not expressed in atleast 5 % of cells
        n_cells = Y.shape[1]
        min_cells = 0.05 * n_cells
        expressed_mask = np.asarray((Y > 0).sum(axis=1)).flatten() >= min_cells
        Y = Y[expressed_mask, :]
        Y_names = np.array(Y_names)[expressed_mask].tolist()
        
        # Filter genes - remove genes with small variance
        # Calculate variance for each gene (rows)
        gene_vars = np.var(Y, axis=1)
        
        # Keep only genes with a variance above a tiny threshold
        mask = gene_vars > 1e-6
        Y = Y[mask, :]
        Y_names = np.array(Y_names)[mask].tolist()
        
        print(f"Remaining genes after variance filtering: {len(Y_names)}")
        
        # Get observations 
        obs = subset.obs.copy()

        # Remove subset from memory
        del subset
        gc.collect()
        
        # X: Design matrix for fixed effects
        obs[fixed_effect] = obs[fixed_effect].astype('category')
        trt_labels = obs[fixed_effect].values
        
        # Get the unique categories to ensure consistent ordering
        categories = obs[fixed_effect].cat.categories.tolist()
        
        # Create the model matrix X (One-Hot Encoding)
        # This creates a column for each category (e.g., A and B)
        X = np.column_stack([trt_labels == cat for cat in categories]).astype(float)
        
        # Create the names for your columns
        X_names = [cat for cat in categories]
        
        print(f"Model Matrix Shape: {X.shape}")
        print(f"Columns: {X_names}")
        
        # Z: design matrix for random effects
        # Identify the column containing your random effect groups (e.g., donor ids)
        # Ensure it is categorical to maintain a stable order
        samples = obs[random_effect].astype('category')
        sample_names = samples.cat.categories.tolist()
        
        # Create the Z matrix (Random Effects Design Matrix)
        # Each column represents one donor; 1.0 if the cell belongs to that sample, else 0.0
        Z = np.column_stack([samples == s for s in sample_names]).astype(float)
        
        # Store names for your records (useful for downstream variance analysis)
        Z_names = [s for s in sample_names]
        
        d = Z.shape[1]
        
        print(f"Z matrix shape: {Z.shape}") # Should be (n_cells, n_samples)
        
        # Ensure X and Z are standard numpy arrays (N_cells x N_features)
        X = np.asarray(X, dtype=np.float64)
        Z = np.asarray(Z, dtype=np.float64)
        
        # Verify dimensions before fitting
        print(f"Y shape (Genes, Cells): {Y.shape}")
        print(f"X shape (Cells, Fixed): {X.shape}")
        print(f"Z shape (Cells, Random): {Z.shape}")
        
        # FLASH-MM using summary statistics
        method = "REML" 
        model_vars = X_names  
        
        # Define contrasts (both directions)
        c1, c2 = X_names[0], X_names[1]
        
        contrast = {
            f"{c1}_vs_{c2}": f"{c1}-{c2}",
            f"{c2}_vs_{c1}": f"{c2}-{c1}"
        }
        
        cm = contrast_matrix(contrast, model_vars)
        
        # Fit Model
        ss = sslmm(X, Y, Z)
        fit = lmm(
            XX=ss["XX"], 
            XY=ss["XY"], 
            ZX=ss["ZX"], 
            ZY=ss["ZY"], 
            ZZ=ss["ZZ"], 
            Ynorm=ss["Ynorm"], 
            n=ss["n"], 
            d=[Z.shape[1]], 
            method=method, 
            Y_names=Y_names, 
            X_names=X_names,
            output_cov=True, 
            output_RE=False
        )
        
        # Run the Statistical Test
        test_output = lmmtest(fit, contrast=cm)
        
        for cname in contrast.keys():
            coef_col = f"{cname}_coef"
            t_stat_col = f"{cname}_t"
            p_val_col = f"{cname}_p"
            
            df_dea = pd.DataFrame({
                'logFC': test_output[coef_col].values,
                't_stat': test_output[t_stat_col].values,
                'p_value': test_output[p_val_col].values
            }, index=Y_names)
            
            # Multiple Testing Correction (FDR)
            mask = ~df_dea['p_value'].isna()
            df_dea['adj_p_val'] = np.nan
            if mask.any():
                df_dea.loc[mask, 'adj_p_val'] = multipletests(
                    df_dea.loc[mask, 'p_value'], method='fdr_bh'
                )[1]
            
            df_dea = df_dea.sort_values('adj_p_val')
            
            print(f"Top Differential Genes for {cname}:")
            print(df_dea.head(10))
            
            results[cname] = df_dea
        print(f"Finished {anno_id} in {time.time() - t0_anno:.2f} sec")
        
# Save
df_all = pd.concat(results, names=["contrast", "gene_symbol"])
df_all = df_all.reset_index(level="contrast").reset_index()

df_all.to_csv(os.path.join(de_overall_dir, f"{prefix}_diff_genes"), index = False)


Processing: alpha
  DE: alpha_up_vs_down
  Paired donors: 146/228
Remaining genes after variance filtering: 10367
Model Matrix Shape: (5807, 2)
Columns: ['alpha_down', 'alpha_up']
Z matrix shape: (5807, 146)
Y shape (Genes, Cells): (10367, 5807)
X shape (Cells, Fixed): (5807, 2)
Z shape (Cells, Random): (5807, 146)
Top Differential Genes for alpha_down_vs_alpha_up:
           logFC     t_stat        p_value      adj_p_val
MT1E   -0.792652 -37.941631  1.319501e-281  1.367927e-277
MT1X   -0.647925 -33.698744  1.559081e-227  8.081494e-224
MT2A   -0.845563 -32.647738  8.566315e-215  2.960233e-211
INS    -0.833574 -27.616811  7.639581e-158  1.979988e-154
PDK4    0.922490  22.746140  8.737143e-110  1.811559e-106
MT1F   -0.325297 -21.543197   4.269520e-99   7.377019e-96
VIM     0.904551  20.902601   1.323211e-93   1.959676e-90
IFITM3 -0.281637 -18.968285   6.802748e-78   8.815511e-75
MT1G   -0.280979 -18.867775   4.099064e-77   4.721667e-74
SPP1   -0.372977 -18.629775   2.786802e-75   2.8890

/work/islet_cartography_scrna/scrna_cartography_milo/lib/python3.13/site-packages/flash_mm/lmm.py:268: RuntimeWarning: invalid value encountered in sqrt
  sebeta.append(np.sqrt(np.diag(covb)))
/work/islet_cartography_scrna/scrna_cartography_milo/lib/python3.13/site-packages/flash_mm/lmmtest.py:76: RuntimeWarning: invalid value encountered in sqrt
  se = np.sqrt(np.diag(contrast.T @ cov_j @ contrast))


Top Differential Genes for myeloid_down_vs_myeloid_up:
             logFC     t_stat       p_value     adj_p_val
RASD1    -0.737755 -11.410328  1.354455e-26  1.201808e-22
HOPX     -0.480212  -9.660651  3.756070e-20  1.666380e-16
FTL       1.290366   8.846405  2.210118e-17  6.536792e-14
SCG5     -1.689860  -8.691948  7.103335e-17  1.575697e-13
CPE      -1.547975  -8.604210  1.370155e-16  2.026231e-13
SEZ6L2   -0.717622  -8.614998  1.264138e-16  2.026231e-13
CTSD      1.704518   8.492171  3.149223e-16  3.991866e-13
SLC22A17 -0.563116  -8.434778  4.809268e-16  5.334079e-13
SCG3     -1.115650  -8.162265  3.493225e-15  3.443932e-12
PCSK1N   -1.677774  -8.090716  5.834524e-15  5.176973e-12
Top Differential Genes for myeloid_up_vs_myeloid_down:
             logFC     t_stat       p_value     adj_p_val
RASD1     0.737755  11.410328  1.354455e-26  1.201808e-22
HOPX      0.480212   9.660651  3.756070e-20  1.666380e-16
FTL      -1.290366  -8.846405  2.210118e-17  6.536792e-14
SCG5      1.689860  

/work/islet_cartography_scrna/scrna_cartography_milo/lib/python3.13/site-packages/flash_mm/lmm.py:268: RuntimeWarning: invalid value encountered in sqrt
  sebeta.append(np.sqrt(np.diag(covb)))
/work/islet_cartography_scrna/scrna_cartography_milo/lib/python3.13/site-packages/flash_mm/lmmtest.py:76: RuntimeWarning: invalid value encountered in sqrt
  se = np.sqrt(np.diag(contrast.T @ cov_j @ contrast))


Top Differential Genes for stellate_activated_down_vs_stellate_activated_up:
           logFC     t_stat       p_value     adj_p_val
CALD1   1.605515  13.170890  5.165144e-36  5.264315e-32
PMS2   -0.016723 -12.871517  1.316170e-34  6.707200e-31
IGFBP7  1.596141  11.913779  2.979701e-30  1.012304e-26
BGN     1.518164  11.669002  3.551389e-29  9.048940e-26
COL3A1  1.542748  11.091810  1.061269e-26  2.163291e-23
COL1A2  1.646549  11.032049  1.892358e-26  3.214485e-23
PDGFRB  1.177332  10.694621  4.750045e-25  6.916066e-22
COL4A1  1.390983  10.602985  1.125426e-24  1.433793e-21
CXCL8  -0.895731 -10.024169  2.300908e-22  2.605651e-19
SPARC   1.358327   9.993812  3.022630e-22  3.067563e-19
Top Differential Genes for stellate_activated_up_vs_stellate_activated_down:
           logFC     t_stat       p_value     adj_p_val
CALD1  -1.605515 -13.170890  5.165144e-36  5.264315e-32
PMS2    0.016723  12.871517  1.316170e-34  6.707200e-31
IGFBP7 -1.596141 -11.913779  2.979701e-30  1.012304e-26
BGN   

/work/islet_cartography_scrna/scrna_cartography_milo/lib/python3.13/site-packages/flash_mm/lmm.py:268: RuntimeWarning: invalid value encountered in sqrt
  sebeta.append(np.sqrt(np.diag(covb)))
/work/islet_cartography_scrna/scrna_cartography_milo/lib/python3.13/site-packages/flash_mm/lmmtest.py:76: RuntimeWarning: invalid value encountered in sqrt
  se = np.sqrt(np.diag(contrast.T @ cov_j @ contrast))


Top Differential Genes for endmt_early_down_vs_other:
            logFC     t_stat       p_value     adj_p_val
SLCO2A1  0.691235  16.472653  2.948788e-58  2.884210e-54
NKX2-3   0.760736  13.528752  1.931088e-40  9.443984e-37
CD320    0.965342  12.921403  3.864668e-37  1.260010e-33
KDR      0.916851  12.322492  5.154567e-34  1.260420e-30
EIF3C    0.265484  11.135254  3.285552e-28  6.427196e-25
SMAD6    0.436389  11.104381  4.576330e-28  7.460180e-25
ATOH8    0.311090  11.087021  5.511632e-28  7.701325e-25
ID1      1.044225  11.047991  8.364637e-28  1.022681e-24
SGK1     0.710233  10.806825  1.069241e-26  1.162027e-23
ENG      1.026387  10.276311  2.429613e-24  2.376404e-21
Top Differential Genes for other_vs_endmt_early_down:
            logFC     t_stat       p_value     adj_p_val
SLCO2A1 -0.691235 -16.472653  2.948788e-58  2.884210e-54
NKX2-3  -0.760736 -13.528752  1.931088e-40  9.443984e-37
CD320   -0.965342 -12.921403  3.864668e-37  1.260010e-33
KDR     -0.916851 -12.322492  5.15456

/work/islet_cartography_scrna/scrna_cartography_milo/lib/python3.13/site-packages/flash_mm/lmm.py:268: RuntimeWarning: invalid value encountered in sqrt
  sebeta.append(np.sqrt(np.diag(covb)))
/work/islet_cartography_scrna/scrna_cartography_milo/lib/python3.13/site-packages/flash_mm/lmmtest.py:76: RuntimeWarning: invalid value encountered in sqrt
  se = np.sqrt(np.diag(contrast.T @ cov_j @ contrast))


Top Differential Genes for mixed_down_vs_mixed_up:
           logFC      t_stat        p_value      adj_p_val
DGKD    0.487158  105.715280   0.000000e+00   0.000000e+00
LBH     1.712375   54.243214  1.138244e-297  5.468693e-294
PLVAP   2.584646   51.072177  1.067941e-278  3.420615e-275
COL4A1  2.439417   50.736006  1.179689e-276  2.833908e-273
COL4A2  2.034546   40.959599  1.097446e-214  2.109072e-211
GNG11   2.086513   40.628714  1.648774e-212  2.640512e-209
ENG     2.158299   39.664400  3.859584e-206  5.298107e-203
RGCC    2.178354   37.179148  1.451357e-189  1.743261e-186
SPARC   1.789909   34.973405  1.079134e-174  1.152155e-171
ID3     1.984975   34.694242  8.388114e-173  8.060139e-170
Top Differential Genes for mixed_up_vs_mixed_down:
           logFC      t_stat        p_value      adj_p_val
DGKD   -0.487158 -105.715280   0.000000e+00   0.000000e+00
LBH    -1.712375  -54.243214  1.138244e-297  5.468693e-294
PLVAP  -2.584646  -51.072177  1.067941e-278  3.420615e-275
COL4A1 -2.439

/work/islet_cartography_scrna/scrna_cartography_milo/lib/python3.13/site-packages/flash_mm/lmm.py:268: RuntimeWarning: invalid value encountered in sqrt
  sebeta.append(np.sqrt(np.diag(covb)))


In [10]:
pd.read_csv(os.path.join(de_overall_dir, f"{prefix}_diff_genes"), index_col = 0)['contrast'].drop_duplicates()

gene_symbol
MT1E                                 alpha_down_vs_alpha_up
MT1E                                 alpha_up_vs_alpha_down
NPY                                    beta_down_vs_beta_up
NPY                                    beta_up_vs_beta_down
RASD1                            myeloid_down_vs_myeloid_up
RASD1                            myeloid_up_vs_myeloid_down
PPY                                     gamma_down_vs_other
PPY                                     other_vs_gamma_down
CALD1      stellate_activated_down_vs_stellate_activated_up
CALD1      stellate_activated_up_vs_stellate_activated_down
ID1                         endothelial_islet_down_vs_other
ID1                         other_vs_endothelial_islet_down
FABP5                                   delta_down_vs_other
FABP5                                   other_vs_delta_down
SLCO2A1                           endmt_early_down_vs_other
SLCO2A1                           other_vs_endmt_early_down
DGKD                        